In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
LATENT_DIM = 2 # Using 2D latent space so we can visualize it directly.
                         # Try 20 later for sharper reconstructions (but no easy 2D plot).
EPOCHS = 20
LEARNING_RATE = 1e-3
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")

# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------
transform = transforms.Compose([transforms.ToTensor()]) # pixel values in [0, 1]

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
class VAE(nn.Module):
    """
    Encoder: image (784) -> hidden -> (mu, logvar) of latent distribution
    Reparameterization: z = mu + std * epsilon
    Decoder: z (latent_dim) -> hidden -> reconstructed image (784)
    """

    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=LATENT_DIM):
        super().__init__()

        # --- Encoder ---
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # --- Decoder ---
        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """Sample z using the reparameterization trick so gradients can flow."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc2(z))
        x_recon = torch.sigmoid(self.fc3(h)) # pixels in [0, 1]
        return x_recon

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar


# ---------------------------------------------------------------------------
# Loss function
# ---------------------------------------------------------------------------
def vae_loss(x_recon, x, mu, logvar):
    """
    VAE loss = Reconstruction loss + KL divergence

    Reconstruction loss: how different is the output from the input?
        (binary cross-entropy works well since pixels are in [0,1])

    KL divergence: pushes the latent distribution q(z|x) toward a standard
    normal N(0, I), which keeps the latent space smooth and continuous
    (this is what lets us interpolate and sample new digits later).
    """
    recon_loss = F.binary_cross_entropy(x_recon, x, reduction="sum")
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_div, recon_loss, kl_div


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def train():
    model = VAE().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = {"total": [], "recon": [], "kl": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, total_recon, total_kl = 0.0, 0.0, 0.0

        for x, _ in train_loader:
            x = x.view(x.size(0), -1).to(DEVICE) # flatten 28x28 -> 784

            optimizer.zero_grad()
            x_recon, mu, logvar = model(x)
            loss, recon_loss, kl_div = vae_loss(x_recon, x, mu, logvar)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_div.item()

        n = len(train_dataset)
        history["total"].append(total_loss / n)
        history["recon"].append(total_recon / n)
        history["kl"].append(total_kl / n)

        print(
            f"Epoch {epoch:2d}/{EPOCHS} | "
            f"Total loss: {total_loss / n:.2f} | "
            f"Recon: {total_recon / n:.2f} | "
            f"KL: {total_kl / n:.2f}"
        )

    return model, history


# ---------------------------------------------------------------------------
# Visualization helpers
# ---------------------------------------------------------------------------
def plot_reconstructions(model, n=10):
    """Show original digits next to their VAE reconstructions."""
    model.eval()
    x, _ = next(iter(test_loader))
    x = x[:n].view(n, -1).to(DEVICE)

    with torch.no_grad():
        x_recon, _, _ = model(x)

    x = x.cpu().view(n, 28, 28)
    x_recon = x_recon.cpu().view(n, 28, 28)

    fig, axes = plt.subplots(2, n, figsize=(n, 2.5))
    for i in range(n):
        axes[0, i].imshow(x[i], cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(x_recon[i], cmap="gray")
        axes[1, i].axis("off")

    axes[0, 0].set_title("Original", loc="left", fontsize=10)
    axes[1, 0].set_title("Reconstructed", loc="left", fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "reconstructions.png"), dpi=150)
    plt.close()
    print("Saved reconstructions.png")


def plot_latent_grid(model, n=20, digit_size=28, range_lim=3):
    """
    Only meaningful when LATENT_DIM == 2.
    Sweeps a grid over the 2D latent space and decodes each point into an
    image, showing how digits morph smoothly across the latent space.
    """
    if LATENT_DIM != 2:
        print("Skipping latent grid plot (only works for LATENT_DIM=2)")
        return

    model.eval()
    grid_x = np.linspace(-range_lim, range_lim, n)
    grid_y = np.linspace(-range_lim, range_lim, n)[::-1] # flip for natural orientation

    canvas = np.zeros((digit_size * n, digit_size * n))

    with torch.no_grad():
        for i, yi in enumerate(grid_y):
            for j, xi in enumerate(grid_x):
                z = torch.tensor([[xi, yi]], dtype=torch.float32).to(DEVICE)
                x_decoded = model.decode(z).cpu().view(digit_size, digit_size).numpy()
                canvas[
                    i * digit_size : (i + 1) * digit_size,
                    j * digit_size : (j + 1) * digit_size,
                ] = x_decoded

    plt.figure(figsize=(8, 8))
    plt.imshow(canvas, cmap="gray")
    plt.title("Digits generated across the 2D latent space")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "latent_grid.png"), dpi=150)
    plt.close()
    print("Saved latent_grid.png")


def plot_random_samples(model, n=64):
    """Generate brand-new digits by sampling random latent vectors."""
    model.eval()
    with torch.no_grad():
        z = torch.randn(n, LATENT_DIM).to(DEVICE)
        samples = model.decode(z).cpu().view(n, 28, 28)

    grid_size = int(n ** 0.5)
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(6, 6))
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i], cmap="gray")
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "random_samples.png"), dpi=150)
    plt.close()
    print("Saved random_samples.png")


def plot_loss_curve(history):
    plt.figure(figsize=(7, 5))
    plt.plot(history["total"], label="Total loss")
    plt.plot(history["recon"], label="Reconstruction loss")
    plt.plot(history["kl"], label="KL divergence")
    plt.xlabel("Epoch")
    plt.ylabel("Loss (per sample)")
    plt.title("VAE Training Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "loss_curve.png"), dpi=150)
    plt.close()
    print("Saved loss_curve.png")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    model, history = train()

    plot_loss_curve(history)
    plot_reconstructions(model, n=10)
    plot_latent_grid(model, n=20)
    plot_random_samples(model, n=64)

    # Save model weights for later reuse
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "vae_mnist.pth"))
    print("Saved model weights to outputs/vae_mnist.pth")

    print("\nDone! Check the 'outputs/' folder for all generated images.")



Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 42.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.13MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.69MB/s]


Epoch  1/20 | Total loss: 190.06 | Recon: 184.27 | KL: 5.80
Epoch  2/20 | Total loss: 167.76 | Recon: 162.45 | KL: 5.30
Epoch  3/20 | Total loss: 163.97 | Recon: 158.64 | KL: 5.33
Epoch  4/20 | Total loss: 162.00 | Recon: 156.62 | KL: 5.38
Epoch  5/20 | Total loss: 160.56 | Recon: 155.11 | KL: 5.46
Epoch  6/20 | Total loss: 159.39 | Recon: 153.89 | KL: 5.50
Epoch  7/20 | Total loss: 158.41 | Recon: 152.83 | KL: 5.58
Epoch  8/20 | Total loss: 157.59 | Recon: 151.97 | KL: 5.62
Epoch  9/20 | Total loss: 156.84 | Recon: 151.16 | KL: 5.68
Epoch 10/20 | Total loss: 156.20 | Recon: 150.48 | KL: 5.72
Epoch 11/20 | Total loss: 155.52 | Recon: 149.76 | KL: 5.76
Epoch 12/20 | Total loss: 154.95 | Recon: 149.13 | KL: 5.82
Epoch 13/20 | Total loss: 154.40 | Recon: 148.54 | KL: 5.86
Epoch 14/20 | Total loss: 153.90 | Recon: 148.00 | KL: 5.90
Epoch 15/20 | Total loss: 153.34 | Recon: 147.40 | KL: 5.94
Epoch 16/20 | Total loss: 152.95 | Recon: 146.98 | KL: 5.97
Epoch 17/20 | Total loss: 152.53 | Recon